# 🤖 AutoTraderX — QLoRA Fine-Tuning

This notebook fine-tunes a language model using **QLoRA** (4-bit quantization + LoRA adapters) for improved financial advisory responses.

**Requirements:**
- Google Colab with **T4 GPU** (free tier works!)
- Upload `training_data.jsonl` from `backend/fine_tuning/`

**What this does:**
1. Loads a base LLM (Llama 3.2 3B Instruct) in 4-bit quantization
2. Trains LoRA adapters on financial advisory data
3. Merges adapters into the base model
4. Pushes the merged model to Hugging Face Hub

---

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers peft bitsandbytes datasets accelerate trl torch huggingface_hub
print("✅ All dependencies installed!")

## Step 2: Check GPU & Login to Hugging Face

You need a HuggingFace token to:
1. Download the base model (gated models like Llama require approval)
2. Push the fine-tuned model to your HF Hub

Get your token at: https://huggingface.co/settings/tokens

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
from huggingface_hub import login

# Login to Hugging Face (paste your token when prompted)
login()

## Step 3: Upload Training Data

Upload `training_data.jsonl` from your local `backend/fine_tuning/` directory.

In [ ]:
from google.colab import files
import json
import os

# Option 1: Upload from local machine
print("📁 Upload your training_data.jsonl file:")
uploaded = files.upload()

DATA_PATH = "training_data.jsonl"

# Verify the data
with open(DATA_PATH, 'r') as f:
    lines = f.readlines()
    sample = json.loads(lines[0])

print(f"\n✅ Loaded {len(lines)} training samples")
print(f"\n📋 Sample instruction: {sample['instruction'][:100]}...")
print(f"📋 Sample output: {sample['output'][:150]}...")

## Step 4: Configuration

Set the base model and training hyperparameters.

In [ ]:
# ═══════════════════════════════════════════════════
# CONFIGURATION — Edit these as needed
# ═══════════════════════════════════════════════════

BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"  # Fast, free, no gating
# Alternative models:
# BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"  # Official (needs HF approval)
# BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"  # Larger, better quality

OUTPUT_DIR = "./autotraderx-qlora-adapter"
MERGED_DIR = "./autotraderx-merged-model"

# Your HuggingFace username (for pushing the model)
HF_USERNAME = "your-hf-username"  # ← CHANGE THIS
HF_REPO_NAME = f"{HF_USERNAME}/autotraderx-financial-advisor"

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 1024
LORA_R = 16           # LoRA rank
LORA_ALPHA = 32       # LoRA alpha
LORA_DROPOUT = 0.05
GRADIENT_ACCUMULATION = 4

print(f"Base model: {BASE_MODEL}")
print(f"LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Training: {EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}")
print(f"Output: {HF_REPO_NAME}")

## Step 5: Load & Prepare Dataset

In [ ]:
from datasets import Dataset

def load_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def format_prompt(sample):
    """Format into Alpaca instruction template."""
    instruction = sample["instruction"]
    context = sample.get("input", "")
    response = sample["output"]

    if context:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Context:\n{context}\n\n"
            f"### Response:\n{response}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{response}"
        )
    return {"text": text}

raw_data = load_jsonl(DATA_PATH)
formatted = [format_prompt(s) for s in raw_data]
dataset = Dataset.from_list(formatted)

print(f"✅ Dataset ready: {len(dataset)} samples")
print(f"\n📋 Sample (first 300 chars):\n{dataset[0]['text'][:300]}...")

## Step 6: Load Base Model with 4-bit Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"⏳ Loading base model: {BASE_MODEL}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✅ Model loaded! Parameters: {model.num_parameters():,}")
print(f"   Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

## Step 7: Configure LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"\n✅ LoRA adapters attached!")
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"   This means we're only training {100*trainable/total:.2f}% of the parameters!")

## Step 8: Train! 🚀

This will take approximately **15-30 minutes** on a T4 GPU with 500 samples.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

print("🚀 Starting QLoRA fine-tuning...")
print(f"   Epochs: {EPOCHS}")
print(f"   Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   Total steps: ~{len(dataset) * EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")
print("="*50)

trainer.train()

print("\n" + "="*50)
print("✅ Fine-tuning complete!")

## Step 9: Save LoRA Adapter

In [ ]:
# Save the LoRA adapter weights
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training config for reference
import json
config = {
    "base_model": BASE_MODEL,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "training_samples": len(dataset),
}
with open(f"{OUTPUT_DIR}/training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ LoRA adapter saved to: {OUTPUT_DIR}")
print(f"   Files: {os.listdir(OUTPUT_DIR)}")

## Step 10: Test the Fine-Tuned Model

In [ ]:
# Test inference with the fine-tuned model
test_prompts = [
    "What do you think about Bitcoin?",
    "Should I buy NVIDIA stock?",
    "Analyze the S&P 500 for me",
]

model.eval()

for prompt in test_prompts:
    full_prompt = f"### Instruction:\n{prompt}\n\n### Response:\n"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part
    response = response.split("### Response:\n")[-1].strip()

    print(f"\n{'='*60}")
    print(f"❓ {prompt}")
    print(f"{'─'*60}")
    print(f"🤖 {response[:500]}")
    print(f"{'='*60}")

## Step 11: Merge LoRA Adapters into Base Model

This creates a standalone model that doesn't need PEFT at inference time.

In [ ]:
from peft import PeftModel

print("⏳ Reloading base model in full precision for merging...")

# Reload base model in float16 (not quantized) for clean merge
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load the LoRA adapter
print("⏳ Loading LoRA adapter...")
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

# Merge LoRA weights into the base model
print("⏳ Merging LoRA weights...")
merged_model = merged_model.merge_and_unload()

# Save the merged model
print(f"⏳ Saving merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\n✅ Merged model saved to: {MERGED_DIR}")
print(f"   This is a standalone model — no PEFT needed at inference!")

## Step 12: Push to Hugging Face Hub

⚠️ **Update `HF_USERNAME` in Step 4** before running this cell!

In [ ]:
if HF_USERNAME == "your-hf-username":
    print("⚠️ Please update HF_USERNAME in Step 4 first!")
    print("   Then re-run this cell.")
else:
    print(f"⏳ Pushing merged model to: {HF_REPO_NAME}...")
    merged_model.push_to_hub(HF_REPO_NAME, private=True)
    tokenizer.push_to_hub(HF_REPO_NAME, private=True)

    # Also push the LoRA adapter separately (much smaller, useful for experiments)
    adapter_repo = f"{HF_USERNAME}/autotraderx-qlora-adapter"
    print(f"⏳ Pushing LoRA adapter to: {adapter_repo}...")
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=adapter_repo,
        repo_type="model",
        create_pr=False,
    )

    print(f"\n✅ Done! Your models are on Hugging Face:")
    print(f"   Merged: https://huggingface.co/{HF_REPO_NAME}")
    print(f"   Adapter: https://huggingface.co/{adapter_repo}")

## Step 13: Download Adapter (Optional)

If you want to download the adapter files to use locally:

In [ ]:
# Download the adapter as a zip file
import shutil

shutil.make_archive("autotraderx-qlora-adapter", "zip", OUTPUT_DIR)
files.download("autotraderx-qlora-adapter.zip")
print("✅ Adapter downloaded! Place in your backend/fine_tuning/ directory.")

---

## 🎉 Done!

Your fine-tuned model is now:
1. ✅ Saved as LoRA adapter in `./autotraderx-qlora-adapter/`
2. ✅ Merged into a standalone model in `./autotraderx-merged-model/`
3. ✅ Pushed to Hugging Face Hub (if you set your username)

### Next Steps:
- Use the model in your AutoTraderX backend by updating `core/llm.py` to load from HF Hub
- Or use the Groq API with the fine-tuned model if you upload it to Groq
- Generate more training data with diverse scenarios for better results